# LlamaIndex: RAG, chatbot e respostas automatizadas em JSON
Processo de RAG (*Retrieval Augmented Generation*) com LlamaIndex: leitura de documentos, divisão em chunks,
embeddings, armazenamento em banco vetorial (ChromaDB), chatbot com memória em Gradio e um pipeline de
recomendação que gera respostas em JSON validadas contra a base de dados.

**Origem:** Alura — projetos de LlamaIndex (chatbot da cafeteria Serenatto e recomendador de livros BotBuscante), com adaptações próprias.

**Dados:**
- pasta `/content/pdf/` com os PDFs do catálogo de cafés (seções 1 a 4);
- `livros_mais_vendidos.csv`, lido do GitHub (https://raw.githubusercontent.com/PRBerlanda/LlamaIndex/refs/heads/Automated-Response-JSON-Formatting/livros_mais_vendidos.csv);
- pasta `/content/artigos/` com textos não estruturados (artigos/resenhas) para ampliar a base (seção 5.3).

**Secrets do Colab:** `GROQ_API` (Groq) e, só para a introdução opcional da seção 1, `OPENAI_API_KEY`.

**Índice**
1. Introdução: RAG mínimo (opcional, OpenAI)
2. Configurando a LLM e o modelo de embeddings
3. RAG com chunks e ChromaDB
4. Chatbot com contexto e interface Gradio
5. Respostas automatizadas em JSON
6. Extra: selecionando chunks relevantes com a LLM

In [ ]:
!pip install -q -U llama-index llama-index-llms-groq llama-index-embeddings-huggingface llama-index-vector-stores-chroma llama-index-readers-file chromadb gradio

import json
import os
import re
import time

import chromadb
import gradio as gr
import pandas as pd
from google.colab import userdata
from llama_index.core import (PromptTemplate, Settings, SimpleDirectoryReader, StorageContext,
                              VectorStoreIndex, get_response_synthesizer)
from llama_index.core.memory import ChatSummaryMemoryBuffer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.query_engine import CustomQueryEngine
from llama_index.core.response_synthesizers import BaseSynthesizer
from llama_index.core.retrievers import BaseRetriever
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.groq import Groq
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore

## 1. Introdução: RAG mínimo (opcional, OpenAI)
A forma mais simples de RAG no LlamaIndex: `SimpleDirectoryReader` lê todos os arquivos de uma pasta,
`VectorStoreIndex` converte os documentos em vetores e os guarda em memória, e `as_query_engine` permite fazer
perguntas sobre eles. Esta seção usa a OpenAI (LLM e embeddings, com custo) e pode ser pulada; o restante do
notebook usa Groq + embeddings do Hugging Face.

In [ ]:
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

llm_openai = OpenAI(model='gpt-5.4-mini', api_key=OPENAI_API_KEY)
embed_openai = OpenAIEmbedding(api_key=OPENAI_API_KEY)

documentos_intro = SimpleDirectoryReader('pdf').load_data()
index_intro = VectorStoreIndex.from_documents(documentos_intro, embed_model=embed_openai)
query_engine_intro = index_intro.as_query_engine(llm=llm_openai)

print(query_engine_intro.query('Quais são os principais assuntos destes documentos?'))

## 2. Configurando a LLM e o modelo de embeddings
- LLM `openai/gpt-oss-20b` servida pela Groq;
- embeddings `intfloat/multilingual-e5-large` (Hugging Face), que funciona bem em português e não tem custo de API.

Os dois são definidos também como padrão global (`Settings`).

In [ ]:
GROQ_API = userdata.get('GROQ_API')

llm = Groq(model='openai/gpt-oss-20b', api_key=GROQ_API, max_tokens=4096)
embed_model = HuggingFaceEmbedding(model_name='intfloat/multilingual-e5-large')

Settings.llm = llm
Settings.embed_model = embed_model

## 3. RAG com chunks e ChromaDB
Objetivos: dividir arquivos PDF, criar embeddings, armazená-los em um banco vetorial e consultar com uma LLM.

### 3.1 Carregamento dos arquivos

In [ ]:
leitor = SimpleDirectoryReader(input_dir='pdf')
print(leitor.input_files)

documents = leitor.load_data()
print(len(documents), 'documentos (páginas)')
print(documents[0].get_metadata_str())

### 3.2 Criando chunks dos documentos
*Chunking* é o processo de dividir um texto extenso em partes menores e mais gerenciáveis, chamadas *chunks*.
Podem ser frases, parágrafos ou trechos de tamanho fixo, dependendo da aplicação. É importante avaliar se os
chunks estão bem delimitados para não haver perda de contexto.

In [ ]:
node_parser = SentenceSplitter(chunk_size=1200)

nodes = node_parser.get_nodes_from_documents(documents, show_progress=True)
print(len(nodes))
print(nodes[0])

### 3.3 Armazenando em um banco vetorial local (ChromaDB)
- Criamos um cliente persistente do ChromaDB (dados em `./chroma_db`) e uma coleção;
- o LlamaIndex calcula os embeddings com o modelo do Hugging Face e grava os vetores, os textos e os metadados
  na coleção, por isso não é preciso configurar uma função de embedding no Chroma;
- se a coleção já tiver vetores (execução anterior), o índice é carregado direto do banco, sem reprocessar.

In [ ]:
chroma_client = chromadb.PersistentClient(path='./chroma_db')
chroma_collection = chroma_client.get_or_create_collection(name='documentos')

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

if chroma_collection.count() == 0:
    index = VectorStoreIndex(nodes, storage_context=storage_context, embed_model=embed_model)
else:
    index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

print('Vetores na coleção:', chroma_collection.count())

### 3.4 Consultando o índice e inspecionando o top-K
O *query engine* busca os chunks mais similares à pergunta (`similarity_top_k`) e os envia como contexto para a LLM.
Também podemos consultar o Chroma diretamente com o embedding da pergunta para ver os chunks recuperados e suas
distâncias.

In [ ]:
query_engine = index.as_query_engine(llm=llm, similarity_top_k=1)

pergunta = 'Quais grãos estão disponíveis?'
response = query_engine.query(pergunta).response
print(response)
print('-' * 30)

# Vetor da pergunta feita à LLM
query_embedding = embed_model.get_query_embedding(pergunta)
print(len(query_embedding), 'dimensões:', query_embedding[:5], '...')
print('-' * 30)

# Chunks mais próximos da pergunta no banco vetorial
resultado = chroma_collection.query(query_embeddings=[query_embedding], n_results=2,
                                    include=['distances', 'documents'])
for distancia, documento in zip(resultado['distances'][0], resultado['documents'][0]):
    print(f'Distância: {distancia:.4f}\n{documento[:300]}...\n')

## 4. Chatbot com contexto e interface Gradio
Fluxo do chat com contexto ampliado:
1. entrada inicial do usuário;
2. consulta/busca por informações relevantes;
3. retorno das informações;
4. entrada inicial + contexto ampliado;
5. resposta gerada pelo modelo.

**Query Engine x Chat Engine**
- *Query Engine*: consultas diretas e pontuais em um índice; ideal para perguntas isoladas e objetivas, mas não
  mantém o contexto da conversa.
- *Chat Engine*: interações dinâmicas e contextualizadas, semelhantes a uma conversa; mantém o histórico e responde
  perguntas de acompanhamento — a escolha para o chatbot da Serenatto, que ajuda o cliente a escolher um café,
  sugere combinações e tira dúvidas sobre métodos de preparo.

### 4.1 Chat engine com memória
`ChatSummaryMemoryBuffer` limita o histórico (aqui 256 tokens), resumindo mensagens antigas para manter o contexto
sem sobrecarregar a janela do modelo. No modo `context`, a cada mensagem o chat engine recupera os textos
relevantes do índice e os coloca como contexto no prompt de sistema.

In [ ]:
memory = ChatSummaryMemoryBuffer(llm=llm, token_limit=256)

chat_engine = index.as_chat_engine(
    chat_mode='context',
    llm=llm,
    memory=memory,
    system_prompt=('''Você é especialista em cafés especiais da Serenatto,
    uma loja online que vende grãos de cafés torrados. Sua função é tirar
    dúvidas de forma simpática e natural sobre os grãos disponíveis''')
)

resposta = chat_engine.chat('Você pode me dar mais detalhes sobre o catuai amarelo?').response
print(resposta)
print('-' * 30)
print(chat_engine.chat_history)
print('-' * 30)
print(memory.get())

### 4.2 Interface com Gradio
O histórico do `gr.Chatbot` usa o formato de mensagens (`role`/`content`); o botão "Limpar" também reseta a
memória do chat engine.

In [ ]:
def converse_com_bot(message, chat_history):
    response = chat_engine.chat(message)
    chat_history = chat_history or []
    chat_history.append({'role': 'user', 'content': message})
    chat_history.append({'role': 'assistant', 'content': response.response})
    return '', chat_history


def resetar_chat():
    chat_engine.reset()
    return []


with gr.Blocks() as demo:
    gr.Markdown('# Chatbot da Serenatto')

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label='Digite a sua mensagem')
    limpar = gr.Button('Limpar')

    msg.submit(converse_com_bot, [msg, chatbot], [msg, chatbot])
    limpar.click(resetar_chat, None, chatbot, queue=False)

demo.launch(share=True)

## 5. Respostas automatizadas em JSON
Recomendador de livros da empresa fictícia Buscante: a LLM só pode recomendar livros que existam na base
(lista de mais vendidos), responde em JSON e a resposta é validada antes de ser formatada para o usuário.

### 5.1 Pergunta sem contexto
Sem RAG a LLM recomenda livros a partir do próprio conhecimento, sem garantia de que estejam no catálogo.

In [ ]:
url = 'https://raw.githubusercontent.com/PRBerlanda/LlamaIndex/refs/heads/Automated-Response-JSON-Formatting/livros_mais_vendidos.csv'
df = pd.read_csv(url)

# O SimpleDirectoryReader lê a pasta "csv"
os.makedirs('csv', exist_ok=True)
df.to_csv('csv/livros_mais_vendidos.csv', index=False)
df.head()

In [ ]:
resposta = llm.complete('Recomende livros de romance')
print(resposta.text)

### 5.2 Criando a base vetorial
A lista de livros é indexada em outra coleção do mesmo cliente Chroma. Reduzimos o tamanho dos chunks
(`chunk_size=512`, `chunk_overlap=20`) para caber na janela de contexto da LLM e medimos o tempo de indexação.

In [ ]:
chroma_collection_livros = chroma_client.get_or_create_collection(name='meus_documentos')
vector_store_livros = ChromaVectorStore(chroma_collection=chroma_collection_livros)
storage_context_livros = StorageContext.from_defaults(vector_store=vector_store_livros)

documentos_livros = SimpleDirectoryReader('csv').load_data()

Settings.chunk_size = 512
Settings.chunk_overlap = 20

start_time = time.time()
if chroma_collection_livros.count() == 0:
    index_livros = VectorStoreIndex.from_documents(
        documentos_livros, storage_context=storage_context_livros, embed_model=embed_model
    )
else:
    index_livros = VectorStoreIndex.from_vector_store(vector_store_livros, embed_model=embed_model)
end_time = time.time()

print(f'Tempo de execução: {end_time - start_time:.1f} segundos')
print(f'Total de IDs: {chroma_collection_livros.count()}')

### 5.3 Inserindo uma nova fonte de dados não estruturada
Documentos novos (pasta `artigos`) podem ser inseridos no índice existente para ampliar o contexto da LLM.

In [ ]:
novos_documentos = SimpleDirectoryReader('artigos').load_data()

for doc in novos_documentos:
    index_livros.insert(doc)

print(f'Total de IDs: {chroma_collection_livros.count()}')

### 5.4 Top-K resultados
Consultando diretamente o Chroma com o embedding da consulta para ver os 10 chunks mais próximos e inspecionando
uma amostra dos registros armazenados.

In [ ]:
query_str = 'Livros de romance'
query_embeded = embed_model.get_query_embedding(query_str)

top_k = chroma_collection_livros.query(query_embeddings=[query_embeded], n_results=10,
                                       include=['distances', 'documents'])
for distancia, documento in zip(top_k['distances'][0], top_k['documents'][0]):
    print(f'{distancia:.4f} | {documento[:120]}')

In [ ]:
doc_ids = chroma_collection_livros.get()['ids']
print('Amostra de IDs:', doc_ids[:5])

amostra = chroma_collection_livros.get(ids=doc_ids[:5], include=['documents', 'metadatas'])
print('Amostra de documentos:', amostra)

### 5.5 Query engine customizado com resposta em JSON
Um `CustomQueryEngine` recupera os chunks relevantes, monta o contexto e pede à LLM **apenas** um objeto JSON no
formato definido no prompt. Como modelos às vezes envolvem o JSON em blocos de código Markdown, `extrair_json`
remove esse invólucro antes do `json.loads`. Depois conferimos se cada título recomendado existe na base.

In [ ]:
retriever = index_livros.as_retriever()
synthesizer = get_response_synthesizer(response_mode='compact', llm=llm)

qa_prompt = PromptTemplate(
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the query.\n"
    "Responda em português.\n"
    "GERE APENAS um objeto JSON válido, e NADA MAIS. Sem texto adicional, explicações ou conversa.\n"
    "O formato JSON DEVE ser exatamente este:\n"
    "{\n"
    "  \"data\":{\n"
    "    \"livros\": [\n"
    "      {\n"
    "        \"titulo\": \"Titulo do livro\",\n"
    "        \"categoria\": \"Categoria do livro\",\n"
    "        \"Justificativa\": \"Porque recomenda\"\n"
    "      }\n"
    "    ]\n"
    "  }\n"
    "}\n"
    "Só recomende livros dessa base de dados para a query: {query_str}\n"
)


class RAGStringQueryEngine(CustomQueryEngine):

    retriever: BaseRetriever
    response_synthesizer: BaseSynthesizer
    llm: Groq
    qa_prompt: PromptTemplate

    def custom_query(self, query_str: str):
        nodes = self.retriever.retrieve(query_str)
        context_str = '\n\n'.join([n.node.get_content() for n in nodes])
        response = self.llm.complete(
            self.qa_prompt.format(context_str=context_str, query_str=query_str)
        )
        return response.text


query_engine_customize = RAGStringQueryEngine(
    retriever=retriever,
    response_synthesizer=synthesizer,
    llm=llm,
    qa_prompt=qa_prompt
)


def extrair_json(texto):
    texto = texto.strip()
    bloco = re.search(r'```(?:json)?\s*(.*?)```', texto, re.DOTALL)
    if bloco:
        texto = bloco.group(1)
    return json.loads(texto)


def livro_na_base(titulo):
    return not df[df['nome'].str.upper().str.contains(titulo.upper(), regex=False)].empty

In [ ]:
resposta = query_engine_customize.query('Quais livros de Romance').response
resposta_json = extrair_json(resposta)

for livro in resposta_json['data']['livros']:
    titulo = livro.get('titulo')
    print(titulo, '->', 'encontrado' if livro_na_base(titulo) else 'não encontrado')

### 5.6 Pipeline completo e publicação de um MVP com Gradio
1. consulta ao query engine customizado; 2. conversão para JSON; 3. validação (todos os títulos precisam existir
na base); 4. formatação da resposta final pela LLM, com a persona de atendente da Buscante.

In [ ]:
# Valida os dados verificando se estão presentes na base de dados
def validando(resposta_json):
    for livro in resposta_json['data']['livros']:
        titulo = livro.get('titulo')
        if not livro_na_base(titulo):
            raise Exception(f'Livro {titulo} não encontrado')


# Melhora a resposta ao usuário integrando a response com a consulta
def formatando_respostas(response, consulta):
    prompt = f'''
    Formate a resposta para o usuario
    Sua persona é um atendente da empresa buscante que vai recomendar livros usando a consulta do usuario como referencia
    E recomendar somente esses livros que estão em response
    consulta: {consulta}
    response: {response}
    '''
    resposta_final = llm.complete(prompt=prompt).text
    return resposta_final


def pipeline_completo(consulta):  # recebe a entrada do usuário
    response = query_engine_customize.query(consulta).response
    resposta_json = extrair_json(response)
    validando(resposta_json)
    return str(formatando_respostas(response, consulta))

In [ ]:
print(pipeline_completo('Quais livros de Romance'))

In [ ]:
demo_livros = gr.Interface(
    fn=pipeline_completo,
    inputs=['text'],
    outputs=['text'],
)

demo_livros.launch(share=True)

### 5.7 Alternativa: interface em Streamlit
O Streamlit não roda dentro de uma célula do Colab: o código vai para um `app.py` (com as funções do pipeline acima)
e é executado no terminal com `streamlit run app.py`. Trecho da interface:

```python
import streamlit as st

st.title("Especialista BotBuscante")
consulta = st.text_input("Digite sua consulta:")
if st.button("Buscar"):
    try:
        resposta = pipeline_completo(consulta)
        st.success("Recomendações:")
        st.write(resposta)
    except Exception as e:
        st.error(f"Erro: {str(e)}")
```

## 6. Extra: selecionando chunks relevantes com a LLM
Alternativa sem banco vetorial: o DataFrame é dividido em blocos de 10 livros, a LLM resume cada bloco, escolhe
(em JSON) os blocos mais relevantes para o pedido e, por fim, recomenda livros apenas desses blocos.

In [ ]:
chunk_size = 10
chunks = [df.iloc[i:i + chunk_size] for i in range(0, len(df), chunk_size)]

summarizing_prompt = """
Resuma o seguinte conjunto de dados de livros em uma descrição breve que destaque os gêneros principais:
Conjunto de dados:
"""
resumos = []

for i, chunk in enumerate(chunks):
    prompt = summarizing_prompt + '\n'.join(chunk['nome'].values)
    resposta = llm.complete(prompt=prompt).text.strip()
    resumos.append({'chunk_index': i, 'resumo': resposta})

In [ ]:
relevance_prompt = """
Com base nos resumos abaixo, selecione os chunks mais relevantes para "Recomende livros de romance".
Retorne a resposta exclusivamente no seguinte formato JSON (não inclua explicações adicionais ou comentários):

{
"relevantes": [índices]
}

Os índices devem corresponder aos números dos chunks mencionados nos resumos.
"""

resumo_texto = '\n'.join([f"Chunk {r['chunk_index']}: {r['resumo']}" for r in resumos])
relevantes_resposta = llm.complete(prompt=relevance_prompt + resumo_texto).text.strip()
relevantes_json = extrair_json(relevantes_resposta)
chunks_relevantes = [chunks[i] for i in relevantes_json['relevantes']]
print('Chunks relevantes:', relevantes_json['relevantes'])

In [ ]:
recomendacao_prompt = """
Recomende livros de romance
Formate em JSON
Responda em JSON
Não escreva mais nada:
{
  "data": {
    "livros": [
      {
        "titulo": "Titulo do livro",
        "categoria": "Categoria do livro",
        "justificativa": "Por que recomenda"
      }
    ]
  }
}
"""

resultados = []

for chunk in chunks_relevantes:
    prompt = recomendacao_prompt + '\n'.join(chunk['nome'].values)
    resposta = llm.complete(prompt=prompt).text.strip()
    try:
        resposta_json = extrair_json(resposta)
        resultados.extend(resposta_json['data']['livros'])
    except json.JSONDecodeError as e:
        print(f'JSONDecodeError: {e}')
        print(f'Resposta da LLM que causou o erro: {resposta}')
    except KeyError as e:
        print(f'KeyError ao acessar a estrutura do JSON: {e}')
        print(f'Resposta da LLM que causou o erro: {resposta}')

pd.DataFrame(resultados)